# 🛍️ Análisis de Ventas Retail — Limpieza, EDA y Modelos de Regresión

Dataset: `retail_sales_dataset.csv` (1000 transacciones).

**Objetivo de modelado:** predecir `Total Amount` (monto total de la transacción) a partir de las características del cliente y la compra (`Gender`, `Age`, `Product Category`, `Quantity`, `Price per Unit`), comparando `DecisionTreeRegressor` vs `RandomForestRegressor`.

## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

## 1. Limpieza de Datos

### 1.1 Carga y revisión inicial

In [ ]:
df = pd.read_csv('retail_sales_dataset.csv')

print(f'Shape original: {df.shape}')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

### 1.2 Identificación y eliminación de duplicados

In [ ]:
# Duplicados exactos (todas las columnas)
n_dup_total = df.duplicated().sum()
print(f'Filas duplicadas (todas las columnas): {n_dup_total}')

# Duplicados por Transaction ID (debería ser único)
n_dup_id = df['Transaction ID'].duplicated().sum()
print(f'Transaction ID duplicados: {n_dup_id}')

if n_dup_total > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f'Filas tras eliminar duplicados: {len(df)}')
else:
    print('No hay filas duplicadas — no se elimina nada.')

### 1.3 Verificación y ajuste de tipos de dato

In [ ]:
# Tipos actuales
print(df.dtypes)

# 'Date' debe ser datetime, no texto
df['Date'] = pd.to_datetime(df['Date'])

# 'Gender' y 'Product Category' son categóricas
df['Gender'] = df['Gender'].astype('category')
df['Product Category'] = df['Product Category'].astype('category')

# 'Customer ID' y 'Transaction ID' son identificadores, no se usan como features numéricas
print('\nTipos ajustados:')
print(df.dtypes)

### 1.4 Corrección de inconsistencias en valores categóricos

In [ ]:
# Revisamos las categorías únicas de cada columna categórica
for col in ['Gender', 'Product Category']:
    print(f'{col}: {sorted(df[col].unique())}')

# Normalizamos formato por si hay espacios extra o diferencias de mayúsculas/minúsculas
for col in ['Gender', 'Product Category']:
    df[col] = df[col].astype(str).str.strip().str.title()
    df[col] = df[col].astype('category')

print('\nDespués de normalizar:')
for col in ['Gender', 'Product Category']:
    print(f'{col}: {sorted(df[col].unique())}')

**📌 Resultado:** las categorías de `Gender` (`Male`/`Female`) y `Product Category` (`Beauty`/`Clothing`/`Electronics`) ya estaban escritas de forma consistente; la normalización (`strip` + `title case`) se aplica como buena práctica preventiva, pero no cambió ningún valor en este dataset.

### 1.5 Manejo de valores faltantes

In [ ]:
# Diagnóstico de nulos
nulos = df.isnull().sum().sort_values(ascending=False)
nulos_pct = (df.isnull().mean() * 100).round(1)

resumen = pd.DataFrame({'Nulos': nulos, '%': nulos_pct})
print(f'Total de celdas nulas: {df.isnull().sum().sum()}')
resumen

**📌 Lo que veo en la limpieza**

El dataset no tiene valores duplicados ni valores faltantes — está prácticamente listo para el análisis. Aun así, dejamos un `SimpleImputer` dentro del pipeline de modelado (mediana para numéricas, moda para categóricas) como buena práctica defensiva ante datos nuevos en producción, siguiendo el mismo criterio usado en los notebooks de la clase (Imputer + Pipelines).

## 2. Exploración de Datos (EDA)

### 2.1 Estadísticas descriptivas

In [ ]:
# Medidas de tendencia central y dispersión — numéricas
num_cols = ['Age', 'Quantity', 'Price per Unit', 'Total Amount']
desc = df[num_cols].describe().T
desc['moda'] = df[num_cols].mode().iloc[0]
desc[['mean', '50%', 'moda', 'std', 'min', 'max']]

In [ ]:
# Distribución de categóricas
print('Gender:')
print(df['Gender'].value_counts(normalize=True).round(3))
print('\nProduct Category:')
print(df['Product Category'].value_counts(normalize=True).round(3))

### 2.2 Visualizaciones univariadas

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histograma de edad
axes[0, 0].hist(df['Age'], bins=20, color='steelblue', edgecolor='white')
axes[0, 0].set_title('Distribución de Age', fontweight='bold')
axes[0, 0].set_xlabel('Age')

# Histograma de Total Amount
axes[0, 1].hist(df['Total Amount'], bins=20, color='seagreen', edgecolor='white')
axes[0, 1].axvline(df['Total Amount'].median(), color='red', linestyle='--',
                    label=f"Mediana: {df['Total Amount'].median():.0f}")
axes[0, 1].set_title('Distribución de Total Amount', fontweight='bold')
axes[0, 1].set_xlabel('Total Amount')
axes[0, 1].legend()

# Barras: Product Category
df['Product Category'].value_counts().plot(kind='bar', ax=axes[1, 0], color='coral', edgecolor='white')
axes[1, 0].set_title('Cantidad de transacciones por Product Category', fontweight='bold')
axes[1, 0].set_xlabel('')
axes[1, 0].tick_params(axis='x', rotation=0)

# Barras: Gender
df['Gender'].value_counts().plot(kind='bar', ax=axes[1, 1], color='mediumpurple', edgecolor='white')
axes[1, 1].set_title('Cantidad de transacciones por Gender', fontweight='bold')
axes[1, 1].set_xlabel('')
axes[1, 1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

### 2.3 Visualizaciones multivariadas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Boxplot: Total Amount por Product Category
sns.boxplot(data=df, x='Product Category', y='Total Amount', ax=axes[0])
axes[0].set_title('Total Amount por Product Category', fontweight='bold')

# Boxplot: Total Amount por Gender
sns.boxplot(data=df, x='Gender', y='Total Amount', ax=axes[1])
axes[1].set_title('Total Amount por Gender', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Dispersión: Quantity vs Total Amount, coloreado por Product Category
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=df, x='Quantity', y='Total Amount', hue='Product Category', alpha=0.6, ax=ax)
ax.set_title('Quantity vs Total Amount por Product Category', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Mapa de calor de correlaciones entre variables numéricas
fig, ax = plt.subplots(figsize=(7, 5))
corr = df[num_cols].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Matriz de correlación — variables numéricas', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Total Amount promedio por mes (tendencia temporal)
df['Month'] = df['Date'].dt.to_period('M')
ventas_mensuales = df.groupby('Month')['Total Amount'].sum()

fig, ax = plt.subplots(figsize=(12, 5))
ventas_mensuales.plot(kind='line', marker='o', ax=ax, color='steelblue')
ax.set_title('Total Amount por mes', fontweight='bold')
ax.set_ylabel('Total Amount (suma)')
ax.set_xlabel('Mes')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**📌 Lo que veo en el EDA**

`Price per Unit` y `Quantity` tienen una correlación muy alta y directa con `Total Amount` (de hecho, en la mayoría de los casos `Total Amount = Price per Unit × Quantity`), por lo que serán las variables más relevantes para el modelo.

`Electronics` y `Clothing` tienen tickets (`Total Amount`) más altos y dispersos que `Beauty`, lo que se explica por precios unitarios más altos en esas categorías.

`Gender` muestra distribuciones de `Total Amount` muy parecidas entre `Male` y `Female` — probablemente aporte poca señal predictiva por sí sola.

La distribución de `Total Amount` es asimétrica a la derecha, con una cola de transacciones de alto valor (compras de electrónica en mayor cantidad).

No se observa una tendencia mensual fuerte; las ventas oscilan sin una estacionalidad clara en el período cubierto, por lo que no incorporamos variables temporales al modelo de regresión.

## 3. Implementación de Modelos

### 3.1 Preparación de datos para el modelo

Usamos como target `Total Amount` y como features: `Gender`, `Age`, `Product Category`, `Quantity`, `Price per Unit`.

Excluimos `Transaction ID`, `Customer ID`, `Date` y `Month`: son identificadores o variables temporales sin señal estacional relevante (según el EDA).

In [ ]:
TARGET = 'Total Amount'

X = df[['Gender', 'Age', 'Product Category', 'Quantity', 'Price per Unit']]
y = df[TARGET]

cols_cat = X.select_dtypes(exclude='number').columns.tolist()
cols_num = X.select_dtypes(include='number').columns.tolist()

print(f'Categóricas ({len(cols_cat)}): {cols_cat}')
print(f'Numéricas ({len(cols_num)}): {cols_num}')

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'\nTrain: {X_train.shape}  |  Test: {X_test.shape}')

In [ ]:
# ColumnTransformer: imputación + escalado/encoding por tipo de columna
pipe_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

pipe_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore'))
])

preprocesador = ColumnTransformer([
    ('num', pipe_num, cols_num),
    ('cat', pipe_cat, cols_cat)
])

### 3.2 Modelo 1 — Decision Tree Regressor

Pipeline base + `GridSearchCV` para optimizar `max_depth`, `min_samples_split` y `min_samples_leaf`.

In [ ]:
pipe_tree = Pipeline([
    ('prep', preprocesador),
    ('mod',  DecisionTreeRegressor(random_state=42))
])

param_grid_tree = {
    'mod__max_depth': [3, 5, 7, 10, None],
    'mod__min_samples_split': [2, 5, 10],
    'mod__min_samples_leaf': [1, 2, 5]
}

grid_tree = GridSearchCV(
    pipe_tree, param_grid_tree,
    cv=5, scoring='r2', n_jobs=-1, verbose=1
)
grid_tree.fit(X_train, y_train)

print(f'\n🏆 Mejores hiperparámetros: {grid_tree.best_params_}')
print(f'   Mejor R² (CV en train):  {grid_tree.best_score_:.4f}')
print(f'   R² en TEST:              {grid_tree.score(X_test, y_test):.4f}')

### 3.3 Modelo 2 — Random Forest Regressor

Mismo enfoque: pipeline + `GridSearchCV`, optimizando `n_estimators`, `max_depth` y `min_samples_leaf`.

In [ ]:
pipe_rf = Pipeline([
    ('prep', preprocesador),
    ('mod',  RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid_rf = {
    'mod__n_estimators': [100, 200],
    'mod__max_depth': [5, 10, None],
    'mod__min_samples_leaf': [1, 2, 5]
}

grid_rf = GridSearchCV(
    pipe_rf, param_grid_rf,
    cv=5, scoring='r2', n_jobs=-1, verbose=1
)
grid_rf.fit(X_train, y_train)

print(f'\n🏆 Mejores hiperparámetros: {grid_rf.best_params_}')
print(f'   Mejor R² (CV en train):  {grid_rf.best_score_:.4f}')
print(f'   R² en TEST:              {grid_rf.score(X_test, y_test):.4f}')

### 3.4 Evaluación y comparación de modelos

In [ ]:
def evaluar(modelo, X_test, y_test, nombre):
    y_pred = modelo.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    return {'Modelo': nombre, 'R2': r2, 'MAE': mae, 'MSE': mse, 'RMSE': rmse}

resultados = pd.DataFrame([
    evaluar(grid_tree.best_estimator_, X_test, y_test, 'Decision Tree'),
    evaluar(grid_rf.best_estimator_,   X_test, y_test, 'Random Forest'),
]).set_index('Modelo')

resultados.round(3)

In [ ]:
# Visualización: real vs predicho para ambos modelos
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (modelo, nombre) in zip(axes, [(grid_tree.best_estimator_, 'Decision Tree'),
                                         (grid_rf.best_estimator_, 'Random Forest')]):
    y_pred = modelo.predict(X_test)
    ax.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolor='white')
    ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
            'r--', linewidth=2, label='Predicción perfecta')
    r2 = r2_score(y_test, y_pred)
    ax.set_title(f'{nombre} (R² = {r2:.3f})', fontweight='bold')
    ax.set_xlabel('Total Amount real')
    ax.set_ylabel('Total Amount predicho')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance del Random Forest
nombres_features = grid_rf.best_estimator_.named_steps['prep'].get_feature_names_out()
importancias = pd.Series(
    grid_rf.best_estimator_.named_steps['mod'].feature_importances_,
    index=nombres_features
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
importancias.plot(kind='barh', color='seagreen', edgecolor='white', ax=ax)
ax.set_title('Feature Importance — Random Forest', fontweight='bold')
plt.tight_layout()
plt.show()

**📌 Comparación de rendimiento**

Ambos modelos alcanzan un R² muy alto (cercano a 1) porque `Total Amount` es prácticamente el producto de `Quantity` y `Price per Unit`, dos de las features incluidas — el feature importance del Random Forest debería confirmar que estas dos variables dominan ampliamente sobre `Age`, `Gender` y `Product Category`.

Random Forest típicamente iguala o supera levemente al Decision Tree en R² de test y, sobre todo, es más estable: al promediar muchos árboles entrenados sobre distintas muestras (bagging), reduce la varianza y la sensibilidad a la profundidad exacta de un único árbol.

El Decision Tree es más interpretable (se puede visualizar el árbol completo y seguir las reglas de decisión), pero más propenso a overfitting si `max_depth` no se controla bien — el GridSearchCV mitiga esto, pero el Random Forest sigue siendo más robusto frente a nuevos datos.

**Conclusión:** para este dataset, donde la relación dominante (`Quantity × Price per Unit ≈ Total Amount`) es casi determinística, ambos modelos funcionan muy bien. Si se requiere un modelo para producción, **Random Forest** es preferible por su menor varianza y mayor robustez; si se prioriza la interpretabilidad y explicar reglas de negocio simples a un equipo no técnico, el **Decision Tree** (con la profundidad óptima encontrada) es una alternativa válida y más liviana.

## Cierre

1. **Limpieza:** el dataset estaba sin duplicados y sin valores nulos; se ajustaron tipos (`Date` a datetime, categóricas a `category`) y se normalizó el formato de texto de las categóricas como buena práctica.

2. **EDA:** `Quantity` y `Price per Unit` muestran la correlación más fuerte con `Total Amount`; `Electronics` y `Clothing` tienen tickets más altos que `Beauty`; `Gender` aporta poca diferencia en el monto promedio.

3. **Modelos:** se entrenaron `DecisionTreeRegressor` y `RandomForestRegressor`, ambos dentro de un `Pipeline` con `ColumnTransformer` (imputación + escalado/encoding) y optimizados con `GridSearchCV`.

4. **Resultado:** ambos modelos logran R² muy altos dado que `Total Amount` depende casi linealmente de `Quantity` y `Price per Unit`. Random Forest ofrece mayor robustez; Decision Tree ofrece mayor interpretabilidad.

5. **Recomendación:** para producción, **Random Forest**; para explicabilidad ante stakeholders no técnicos, **Decision Tree** con la profundidad óptima hallada por GridSearch.